In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:
races_df = spark.read.\
  option("header",True).\
    option("inferSchema",True).\
      csv("abfss://raw@databricksrg2026.dfs.core.windows.net/races.csv")



In [0]:
races_df.show()

In [0]:
display(races_df)

In [0]:
races_df.printSchema()

In [0]:
from pyspark.sql.types import StructType , StructField , IntegerType , StringType , DoubleType


In [0]:
races_schema = StructType(fields = [
  StructField("raceId",IntegerType(),False),
  StructField("year",IntegerType(),True),
  StructField("round",IntegerType(),True),
  StructField("circuitId", IntegerType(),False),
  StructField("name", StringType(),True),
  StructField("date", StringType(),True),
  StructField("time", StringType(),True),
  StructField("url", StringType(),True)
])


In [0]:
display(races_schema)

In [0]:
races_selected_df = races_df.select("raceId","year","round","circuitId","name")

In [0]:
races_renamed_df = races_selected_df.withColumnRenamed("raceId","race_id")\
  .withColumnRenamed("year","race_year")\
  .withColumnRenamed("circuitId","circuit_id")


In [0]:
display(races_renamed_df)

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
races_final_df = races_renamed_df.withColumn("ingestion_date",current_timestamp())


In [0]:
display(races_final_df)

In [0]:
races_final_df.write \
    .mode("overwrite") \
    .parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/races")

partition by race_

In [0]:
races_final_df.write.mode("overwrite").partitionBy('race_year').parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/races")